# 🛒 E-Commerce Product Recommendation Engine

This notebook demonstrates a complete recommendation system with:
- **Content-Based Filtering** (TF-IDF + Cosine Similarity)
- **Collaborative Filtering** (User-based, Item-based, SVD)
- **Hybrid Model** (weighted blend of CB + SVD)
- **Cold-Start Fallback** (popularity-based)
- **Evaluation** (Precision@K, Recall@K, NDCG@K, HitRate@K)


## 0. Setup & Imports

In [ ]:
import sys, warnings, json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

warnings.filterwarnings('ignore')

# ensure project root is on sys.path
ROOT = Path('.').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# style
plt.rcParams.update({
    'figure.facecolor': '#0a0e1a',
    'axes.facecolor':   '#111827',
    'axes.edgecolor':   '#374151',
    'axes.labelcolor':  '#9ca3af',
    'xtick.color':      '#9ca3af',
    'ytick.color':      '#9ca3af',
    'text.color':       '#f1f5f9',
    'grid.color':       '#1f2937',
    'grid.linestyle':   '--',
    'font.family':      'DejaVu Sans',
    'figure.dpi':       110,
})

PALETTE = ['#6366f1','#f59e0b','#10b981','#ef4444','#3b82f6','#a855f7']

print('✓ Imports OK')

## 1. Data Generation

In [ ]:
from database.seed import initialise_database, load_dataframes
from data.data_generator import load_dataset

# Load directly from SQLite database (auto-initialises & seeds if empty)
initialise_database()
products, users, interactions = load_dataframes()

print(f'\nProducts shape    : {products.shape}')
print(f'Users shape       : {users.shape}')
print(f'Interactions shape: {interactions.shape}')
products.head(3)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.patch.set_facecolor('#0a0e1a')

# category bar chart
cat_counts = products['category'].value_counts()
bars = axes[0].barh(cat_counts.index[::-1], cat_counts.values[::-1],
                    color=PALETTE * 2, edgecolor='none', height=0.65)
axes[0].set_title('Products per Category', pad=10, fontsize=13, fontweight='bold')
axes[0].set_xlabel('Count')
axes[0].grid(axis='x', alpha=0.3)
axes[0].tick_params(axis='both', labelsize=9)

# interaction type donut
itype = interactions['interaction_type'].value_counts()
wedges, texts, autotexts = axes[1].pie(
    itype.values, labels=itype.index,
    colors=PALETTE[:len(itype)],
    autopct='%1.1f%%', startangle=90,
    wedgeprops=dict(width=0.55, edgecolor='#0a0e1a', linewidth=2),
    textprops=dict(color='#f1f5f9', fontsize=9),
)
for at in autotexts:
    at.set_fontsize(8)
axes[1].set_title('Interaction Type Distribution', pad=10, fontsize=13, fontweight='bold')

plt.tight_layout(pad=2)
plt.show()

## 2. Train / Test Split (Temporal 80/20)

In [ ]:
from evaluation.metrics import temporal_train_test_split

train, test = temporal_train_test_split(interactions, test_ratio=0.20)
print(f'Train: {len(train):,} interactions')
print(f'Test : {len(test):,} interactions')

## 3. Train All Models

### 3a. Content-Based Filtering (TF-IDF + Cosine Similarity)

In [ ]:
from models.content_based import ContentBasedRecommender

cb_model = ContentBasedRecommender(max_features=5_000)
cb_model.fit(products)
print('✓ Content-Based model trained')

### 3b. Collaborative Filtering — User-based, Item-based, SVD

In [ ]:
from models.collaborative import UserBasedCF, ItemBasedCF, SVDRecommender

user_cf = UserBasedCF(k_neighbors=30)
user_cf.fit(train)
print('✓ User-based CF trained')

item_cf = ItemBasedCF(k_neighbors=30)
item_cf.fit(train)
print('✓ Item-based CF trained')

svd_model = SVDRecommender(n_factors=50)
svd_model.fit(train)
print('✓ SVD Recommender trained')

### 3c. Hybrid Model (Content-Based + SVD)

In [ ]:
from models.hybrid import HybridRecommender

hybrid = HybridRecommender(alpha=0.4, n_factors=50)
hybrid.fit(products, train)
print('✓ Hybrid model trained')

### 3d. Cold-Start Fallback (Popularity-Based)

In [ ]:
from utils.cold_start import ColdStartRecommender

cold_start = ColdStartRecommender(min_interactions=3, recency_halflife=90)
cold_start.fit(products, train)
print('✓ Cold-Start recommender fitted')

## 4. Demo — Similar Products

In [ ]:
def show_product(pid, products_df):
    row = products_df[products_df['product_id'] == pid].iloc[0]
    return f"[{pid}] {row['title']} | {row['category']} | ${row['price']:.2f}"

# pick a sample product
sample_pid = products.sample(1, random_state=7)['product_id'].values[0]
print('Query Product:')
print(' ', show_product(sample_pid, products))
print()

print('── Content-Based Similar Products ──')
for i, (pid, score) in enumerate(cb_model.get_similar_products(sample_pid, n=5), 1):
    print(f'  {i}. {show_product(pid, products)}  (score={score:.4f})')

print()
print('── Item-CF Similar Products ──')
for i, (pid, score) in enumerate(item_cf.get_similar_items(sample_pid, n=5), 1):
    print(f'  {i}. {show_product(pid, products)}  (score={score:.4f})')

print()
print('── SVD Latent-Space Similar Products ──')
for i, (pid, score) in enumerate(svd_model.get_similar_items(sample_pid, n=5), 1):
    print(f'  {i}. {show_product(pid, products)}  (score={score:.4f})')

## 5. Demo — Personalised Recommendations per User

In [ ]:
def print_recs(recs, products_df, label, n=5):
    print(f'\n── {label} ──')
    if not recs:
        print('  (no recs returned)')
        return
    for i, (pid, score) in enumerate(recs[:n], 1):
        row = products_df[products_df['product_id'] == pid]
        if row.empty: continue
        r = row.iloc[0]
        print(f'  {i}. [{pid}] {r["title"]} | {r["category"]} | ${r["price"]:.2f}  (score={score:.4f})')

# Select demo users
counts = train['user_id'].value_counts()
heavy_uid = counts.index[0]             # most interactions
light_uid = counts[counts <= 3].index[0]  # few interactions
all_train_uids = set(train['user_id'])
new_uid = next(u for u in users['user_id'] if u not in all_train_uids)  # zero interactions

print(f'Heavy user: {heavy_uid}  ({counts[heavy_uid]} interactions)')
print(f'Light user: {light_uid}  ({counts[light_uid]} interactions)')
print(f'New user  : {new_uid}   (0 interactions)')

In [ ]:
print(f'\n{'='*65}')
print(f'HEAVY USER: {heavy_uid}')
print(f'{'='*65}')
print_recs(cb_model.get_recommendations(heavy_uid, train, n=5), products, 'Content-Based')
print_recs(user_cf.get_recommendations(heavy_uid, n=5),         products, 'User-CF')
print_recs(svd_model.get_recommendations(heavy_uid, n=5),       products, 'SVD')
print_recs(hybrid.get_recommendations(heavy_uid, n=5),          products, 'Hybrid')

In [ ]:
print(f'\n{'='*65}')
print(f'LIGHT USER: {light_uid}')
print(f'{'='*65}')
print_recs(cb_model.get_recommendations(light_uid, train, n=5), products, 'Content-Based')
print_recs(user_cf.get_recommendations(light_uid, n=5),         products, 'User-CF')
print_recs(svd_model.get_recommendations(light_uid, n=5),       products, 'SVD')
print_recs(hybrid.get_recommendations(light_uid, n=5),          products, 'Hybrid')

In [ ]:
print(f'\n{'='*65}')
print(f'NEW USER (Cold Start): {new_uid}')
print(f'{'='*65}')
pref_row = users[users['user_id'] == new_uid]
prefs = pref_row['preferred_categories'].values[0].split(',') if not pref_row.empty else None
print(f'Preferred categories: {prefs}')
print_recs(cold_start.get_recommendations(new_uid, n=5, preferred_categories=prefs),
           products, 'Cold-Start (Popularity)')

## 6. Evaluation — Precision@K, Recall@K, NDCG@K, HitRate@K

In [ ]:
from evaluation.metrics import compare_models

models_to_eval = {
    'Popularity':    lambda uid, n: cold_start.get_popular_products(n),
    'Content-Based': lambda uid, n: cb_model.get_recommendations(uid, train, n),
    'User-CF':       lambda uid, n: user_cf.get_recommendations(uid, n),
    'Item-CF':       lambda uid, n: item_cf.get_recommendations(uid, n),
    'SVD':           lambda uid, n: svd_model.get_recommendations(uid, n),
    'Hybrid':        lambda uid, n: hybrid.get_recommendations(uid, n),
}

results_df = compare_models(
    models_to_eval, train, test,
    k_values=[5, 10, 20],
    max_users=150,
)
results_df

### 6a. Pivot table — easy comparison at K=10

In [ ]:
k10 = results_df[results_df['K'] == 10].set_index('Model')[['Precision','Recall','NDCG','HitRate']]
k10 = k10.sort_values('NDCG', ascending=False)
k10.style \
    .background_gradient(cmap='Purples', subset=['Precision']) \
    .background_gradient(cmap='Blues',   subset=['Recall']) \
    .background_gradient(cmap='Greens',  subset=['NDCG']) \
    .background_gradient(cmap='Oranges', subset=['HitRate']) \
    .format('{:.4f}') \
    .set_caption('Evaluation Results @ K=10')

### 6b. Visualise Metrics Across K Values

In [ ]:
metrics = ['Precision', 'Recall', 'NDCG', 'HitRate']
k_vals  = sorted(results_df['K'].unique())
models  = results_df['Model'].unique().tolist()

fig, axes = plt.subplots(2, 2, figsize=(15, 10))
fig.patch.set_facecolor('#0a0e1a')
fig.suptitle('Recommendation Model Comparison', fontsize=16, fontweight='bold',
             color='#f1f5f9', y=1.01)

for ax, metric in zip(axes.flat, metrics):
    for i, model in enumerate(models):
        sub = results_df[results_df['Model'] == model].sort_values('K')
        ax.plot(sub['K'], sub[metric],
                marker='o', linewidth=2.5, markersize=7,
                color=PALETTE[i % len(PALETTE)], label=model)

    ax.set_title(f'{metric} @ K', fontsize=12, fontweight='bold', pad=8)
    ax.set_xlabel('K')
    ax.set_ylabel(metric)
    ax.set_xticks(k_vals)
    ax.legend(fontsize=8, framealpha=0.2)
    ax.grid(alpha=0.3)

plt.tight_layout(pad=2.5)
plt.show()

### 6c. Grouped Bar Chart — All Models @ K=10

In [ ]:
k10_reset = k10.reset_index()
model_names = k10_reset['Model'].tolist()
x = np.arange(len(model_names))
width = 0.2

fig, ax = plt.subplots(figsize=(14, 6))
fig.patch.set_facecolor('#0a0e1a')

for i, metric in enumerate(metrics):
    vals = k10_reset[metric].values
    bars = ax.bar(x + i * width - 1.5 * width, vals, width,
                  label=metric, color=PALETTE[i], edgecolor='none',
                  alpha=0.88)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.002,
                f'{v:.3f}', ha='center', va='bottom', fontsize=7, color='#94a3b8')

ax.set_title('All Metrics @ K=10 by Model', fontsize=14, fontweight='bold', pad=10)
ax.set_xticks(x)
ax.set_xticklabels(model_names, fontsize=10)
ax.set_ylabel('Score')
ax.legend(fontsize=10, framealpha=0.2)
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(k10_reset[metrics].values.max() * 1.15, 0.05))

plt.tight_layout()
plt.show()

## 7. Generate HTML Dashboard

In [ ]:
# Run the full main pipeline which also generates the HTML report
# (Skip data generation — reuse already-generated data)
import importlib, main as m
importlib.reload(m)

demo_users = [
    ('Heavy user', heavy_uid, False),
    ('Light user', light_uid, False),
    ('New user',   new_uid,   True),
]

report_path = ROOT / 'dashboard' / 'report.html'
report_path.parent.mkdir(parents=True, exist_ok=True)

m._generate_html_report(
    report_path=report_path,
    products=products, users=users, interactions=interactions,
    train=train, test=test,
    results_df=results_df,
    cb_model=cb_model, svd_model=svd_model,
    hybrid=hybrid, cold_start=cold_start,
    demo_users=demo_users, demo_train=train,
)

print(f'✓ Dashboard saved → {report_path}')
from IPython.display import HTML
HTML(f'<a href="{report_path}" target="_blank">📊 Open Dashboard in New Tab</a>')

## Summary

| Model | Approach | Cold-Start Safe? |
|---|---|---|
| **Content-Based** | TF-IDF + Cosine on product features | ✅ (new user: no recs) |
| **User-CF** | User-user cosine similarity on rating matrix | ❌ |
| **Item-CF** | Item-item cosine similarity on rating matrix | ❌ |
| **SVD** | Matrix factorization (latent factors) | ❌ |
| **Hybrid** | α×CB + (1-α)×SVD blended scores | Partial |
| **Cold-Start** | Recency-weighted popularity (global + per category) | ✅ |
